# 🔬 SkinCheck AI - Multi-Task Skin Condition Trainer
### 🚀 Deteksi Visual: Jerawat (Acne), Kemerahan (Redness), Hyperpigmentasi, & Tekstur Kulit

Selamat datang di pipeline pelatihan AI **SkinCheck**! Notebook ini dirancang khusus untuk memenuhi alur kerja end-to-end:
1. 🔑 **Auto Kaggle Connection**: Otomatis terhubung dengan kredensial Kaggle Anda (`satriafibisaputra`).
2. 📥 **Universal Dataset Downloader**: Cukup paste URL dataset Kaggle apa saja, sistem langsung mengunduh dan mengekstrak.
3. 📦 **Sistem Penimbunan Dataset (Dataset Hoarder & Deduplicator)**: Anda bisa berulang kali mengunduh berbagai dataset, membersihkannya, dan menimbunnya ke dalam satu Master Dataset tanpa duplikasi (verifikasi MD5 checksum).
4. 🔄 **Multi-Angle & Distance Augmentation (Pose Invariant)**: Mensimulasikan pemindaian wajah multi-posisi (hadap kanan, kiri, atas, bawah, jarak dekat, dan jarak jauh) sehingga AI terlatih menangani pemindaian dinamis.
5. 🧠 **Multi-Task Deep Learning Network**: Arsitektur backbone canggih (MobileNetV3 / EfficientNet) dengan 4 output head simultan (Skor 0-100 per kondisi).
6. 🔁 **Continuous Training / Checkpoint Resume**: Latih model kapan saja. Jika data dirasa kurang, tambah dataset baru dan lanjutkan training tanpa mengulang dari nol.
7. 🧪 **In-Colab Interactive Testing**: Uji coba gambar dan simulasi multi-frame burst scan langsung di dalam Colab.
8. 📦 **One-Click Export ke Antigravity**: Ekspor model ke `.onnx`, `.pt`, dan `model_metadata.json` siap pakai untuk aplikasi web SkinCheck Anda.

---
> 💡 **Petunjuk Penggunaan:**
> - Pastikan GPU Colab sudah aktif: Klik menu **Runtime** -> **Change runtime type** -> Pilih **T4 GPU** -> Simpan.
> - Jalankan sel secara berurutan dengan tombol Play `[▶]` atau tekan `Shift + Enter`.


## 🛠️ Tahap 1: Persiapan Environment & Otentikasi Kaggle API
Sel ini akan menginstal modul yang diperlukan dan mengonfigurasi kredensial Kaggle Anda (`satriafibisaputra`) secara otomatis.


In [ ]:
# 1. Setup Dependencies & Autentikasi Kaggle
import os
import sys
import json
import time
import shutil
import hashlib
from pathlib import Path

print("📦 Memeriksa dan menginstal dependencies...")
!pip install -q kaggle albumentations onnx onnxruntime timm torchvision opencv-python matplotlib seaborn

# Konfigurasi Kredensial Kaggle User
KAGGLE_USERNAME = "satriafibisaputra"
KAGGLE_KEY = os.getenv("KAGGLE_KEY", "YOUR_KAGGLE_KEY_HERE")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

credentials = {
    "username": KAGGLE_USERNAME,
    "key": KAGGLE_KEY
}

with open(kaggle_json, "w") as f:
    json.dump(credentials, f)

os.chmod(kaggle_json, 0o600)
print(f"✅ Kaggle credentials terkonfigurasi untuk akun: '{KAGGLE_USERNAME}'")

# Inisialisasi Struktur Folder SkinCheck di Google Colab
BASE_DIR = Path("/content/skincheck_data")
INCOMING_DIR = BASE_DIR / "incoming"           # Folder download sementara
MASTER_DIR = BASE_DIR / "master_dataset"       # Gudang penimbunan dataset
CHECKPOINT_DIR = BASE_DIR / "checkpoints"      # Penyimpanan bobot model
EXPORT_DIR = BASE_DIR / "exports"              # Hasil ekspor untuk Antigravity

# Kategori Utama SkinCheck
CATEGORIES = ["acne", "redness", "hyperpigmentation", "texture", "normal_skin"]

for cat in CATEGORIES:
    (MASTER_DIR / cat).mkdir(parents=True, exist_ok=True)

INCOMING_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("📁 Direktori kerja SkinCheck siap di:", BASE_DIR)
!kaggle datasets list --max-size 1000 -s "skin" | head -n 5
print("🚀 Koneksi ke Kaggle API Berhasil!")


## 📥 Tahap 2: Download Dataset dari Kaggle (Cukup Paste Link/URL)
Anda dapat memasukkan **URL lengkap dari browser** atau **slug dataset** Kaggle pada kolom di bawah.

Contoh yang didukung:
- `https://www.kaggle.com/datasets/subirkesharwani/skin-disease-dataset`
- `https://www.kaggle.com/datasets/arashnic/skin-burn-dataset`
- `shilpams/skin-burn-dataset`
- `ismailpromus/skin-diseases-image-dataset`


In [ ]:
# @title 🔗 Masukkan Link Dataset Kaggle
import re
import zipfile
import tarfile

DATASET_URL_OR_SLUG = "https://www.kaggle.com/datasets/subirkesharwani/skin-disease-dataset" # @param {type:"string"}

def extract_dataset_slug(url_or_slug: str) -> str:
    """Mengekstrak owner/dataset-name dari berbagai variasi URL Kaggle."""
    url_or_slug = url_or_slug.strip()
    match = re.search(r"kaggle\.com/datasets/([^/\s\?#]+/[^/\s\?#]+)", url_or_slug)
    if match:
        return match.group(1)
    parts = url_or_slug.split("/")
    if len(parts) >= 2:
        return f"{parts[-2]}/{parts[-1]}"
    return url_or_slug

dataset_slug = extract_dataset_slug(DATASET_URL_OR_SLUG)
print(f"🎯 Target Dataset Slug: {dataset_slug}")

# Folder download spesifik untuk dataset ini di dalam incoming/
slug_folder_name = dataset_slug.replace("/", "_")
current_download_dir = INCOMING_DIR / slug_folder_name
current_download_dir.mkdir(parents=True, exist_ok=True)

print(f"⬇️ Memulai pengunduhan dari Kaggle ke: {current_download_dir} ...")
download_cmd = f"kaggle datasets download -d {dataset_slug} -p \"{current_download_dir}\" --unzip"
ret = os.system(download_cmd)

# Fallback: jika kaggle cli tidak auto-unzip semua arsip
for archive in current_download_dir.glob("*.zip"):
    print(f"📦 Mengekstrak sisa ZIP: {archive.name}...")
    try:
        with zipfile.ZipFile(archive, 'r') as zip_ref:
            zip_ref.extractall(current_download_dir)
        archive.unlink() # Hapus zip setelah diekstrak untuk menghemat disk Colab
    except Exception as e:
        print(f"⚠️ Gagal ekstrak {archive.name}: {e}")

total_files = len(list(current_download_dir.rglob("*.*")))
print(f"✅ Download & Ekstraksi Selesai! Ditemukan {total_files} file di dalam: {slug_folder_name}")


## 📦 Tahap 3: Pembersih Cerdas & Sistem Penimbunan Dataset (Dataset Accumulator)
Fitur ini memungkinkan Anda untuk **menimbun dataset sebanyak-banyaknya secara berulang**.

Setiap kali Anda mendownload dataset baru di Tahap 2, jalankan sel ini:
- 🛡️ **Deduplikasi**: Memeriksa sidik jari hash MD5 setiap foto. Foto yang sama tidak akan dimasukkan dobel.
- 🧹 **Pembersihan**: Membuang file corrupt, file bukan gambar, dan gambar beresolusi rusak/terlalu kecil (<64px).
- 🏷️ **Smart Auto-Categorizer**: Mengenali nama folder/file dan mengelompokkannya secara otomatis ke 4 indikator utama:
  - `acne` (Jerawat, Comedo, Pimple, Pustule, Papule, Nodule)
  - `redness` (Erythema, Rosacea, Kemerahan, Merah, Ruam)
  - `hyperpigmentation` (Melasma, Dark Spot, Flek, Sunspot, Pigmentasi)
  - `texture` (Pori besar, Kerutan, Roughness, Kulit Kasar)
  - `normal_skin` (Kulit bersih/normal/baseline sehat)
- 📊 Menyimpan ke **Master Repository** dan mencatat manifest inventaris.


In [ ]:
# 3. Smart Dataset Cleaner & Accumulator Pipeline
import cv2
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

MANIFEST_FILE = BASE_DIR / "dataset_manifest.json"

# Load atau inisialisasi manifest penimbunan
if MANIFEST_FILE.exists():
    with open(MANIFEST_FILE, "r") as f:
        manifest = json.load(f)
else:
    manifest = {
        "total_images": 0,
        "hashes": {}, # md5 -> {"category": cat, "file": path, "source": slug}
        "categories": {cat: 0 for cat in CATEGORIES},
        "datasets_ingested": []
    }

def get_file_md5(filepath):
    """Hitung MD5 checksum untuk mendeteksi gambar duplikat."""
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Pola kata kunci untuk pemetaan otomatis ke 4 kondisi SkinCheck
CATEGORY_KEYWORDS = {
    "acne": ["acne", "jerawat", "comedo", "pimple", "papule", "pustule", "nodule", "whitehead", "blackhead", "cystic"],
    "redness": ["redness", "erythema", "rosacea", "merah", "kemerahan", "rash", "eczema", "blush", "dermatitis", "vascular"],
    "hyperpigmentation": ["hyperpigmentation", "pigment", "melasma", "dark_spot", "flek", "sunspot", "freckles", "lentigo", "discoloration", "nevus", "melanocytic"],
    "texture": ["texture", "wrinkle", "pore", "pori", "rough", "kasar", "keriput", "sagging", "lines", "keratosis", "atopic"],
    "normal_skin": ["normal", "clear", "healthy", "clean", "sehat", "mulus", "baseline", "flawless"]
}

def detect_category(file_path: Path) -> str:
    """Mendeteksi kategori berdasarkan nama folder hirarkis dan nama file."""
    path_str = str(file_path).lower().replace("\\", "/")
    
    for cat, keywords in CATEGORY_KEYWORDS.items():
        for kw in keywords:
            if re.search(r'[/_\-]' + re.escape(kw) + r'[/_\-.]', path_str) or (kw in path_str):
                return cat
                
    # Fallback default: jika dataset kulit umum, masukkan ke kategori yang paling sedikit agar seimbang
    counts = {cat: manifest["categories"][cat] for cat in CATEGORIES if cat != "normal_skin"}
    min_cat = min(counts, key=counts.get)
    return min_cat

print("🔍 Memulai pemindaian dan pembersihan berkas di incoming/ ...")
valid_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
new_added = 0
skipped_duplicate = 0
skipped_corrupt = 0

all_incoming_files = [f for f in INCOMING_DIR.rglob("*.*") if f.suffix.lower() in valid_extensions]
print(f"📊 Menemukan {len(all_incoming_files)} file gambar untuk diproses.")

for idx, img_path in enumerate(all_incoming_files):
    try:
        # Validasi pembacaan gambar
        with Image.open(img_path) as img:
            img.verify()
        
        # Validasi resolusi & format dengan OpenCV
        cv_img = cv2.imread(str(img_path))
        if cv_img is None:
            skipped_corrupt += 1
            continue
            
        h, w = cv_img.shape[:2]
        if h < 64 or w < 64: # Buang gambar terlalu kecil
            skipped_corrupt += 1
            continue

        # Cek Deduplikasi MD5
        file_hash = get_file_md5(img_path)
        if file_hash in manifest["hashes"]:
            skipped_duplicate += 1
            continue

        # Klasifikasikan kategori
        cat = detect_category(img_path)
        
        # Simpan ke Master Repository
        target_filename = f"{cat}_{file_hash[:12]}.jpg"
        target_path = MASTER_DIR / cat / target_filename
        
        # Normalisasi dan simpan sebagai RGB standard JPG
        cv2.imwrite(str(target_path), cv_img, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

        # Catat ke manifest
        manifest["hashes"][file_hash] = {
            "category": cat,
            "filename": target_filename,
            "source_dataset": dataset_slug
        }
        manifest["categories"][cat] += 1
        manifest["total_images"] += 1
        new_added += 1

    except Exception as e:
        skipped_corrupt += 1

if dataset_slug not in manifest["datasets_ingested"]:
    manifest["datasets_ingested"].append(dataset_slug)

# Simpan Manifest Terkini
with open(MANIFEST_FILE, "w") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "="*50)
print("🎉 PROSES PENIMBUNAN DATASET SELESAI!")
print(f"✅ Gambar baru yang ditambahkan: {new_added}")
print(f"🔁 Gambar duplikat yang dilewati: {skipped_duplicate}")
print(f"⚠️ Gambar corrupt/rusak yang dibuang: {skipped_corrupt}")
print(f"📚 Total Dataset yang sudah ditimbun ({len(manifest['datasets_ingested'])}):")
for d in manifest["datasets_ingested"]:
    print(f"   - {d}")
print("="*50)

# Visualisasikan Ringkasan Gudang Dataset
cat_names = list(manifest["categories"].keys())
cat_counts = list(manifest["categories"].values())

plt.figure(figsize=(9, 4))
bars = plt.bar(cat_names, cat_counts, color=['#e74c3c', '#e67e22', '#8e44ad', '#3498db', '#2ecc71'])
plt.title(f"📦 Inventaris Gudang Master Dataset SkinCheck (Total: {manifest['total_images']} Foto)", fontsize=12, fontweight='bold')
plt.xlabel("Kondisi Kulit")
plt.ylabel("Jumlah Foto Bersih")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{int(yval)}", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()


## 🔄 Tahap 4: Multi-Angle, Multi-Distance & Multi-Pose Augmentation
User tidak hanya diam memotret 1 pose, tetapi melakukan pemindaian wajah aktif:
- **Hadap Kanan & Hadap Kiri (Yaw Rotation & Perspective Warp)**
- **Hadap Atas & Hadap Bawah (Pitch Tilt & Vertical Compression)**
- **Jarak Dekat vs Jarak Jauh (Macro Zoom vs Full Face Scale)**
- **Variasi Pencahayaan & Getaran Gerak Kamera**

Pipeline di bawah mensimulasikan ribuan variasi sudut dan jarak secara dinamis saat model dilatih, membuat AI sangat tangguh (robust) terhadap sudut pandang apa pun!


In [ ]:
# 4. Multi-Angle & Distance Invariant Augmentation Pipeline
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Simulasi Rotasi Sudut Wajah & Jarak Kamera Dinamis
class SkinCheckMultiAngleTransforms:
    def __init__(self, is_train=True):
        if is_train:
            self.transform = T.Compose([
                # 1. Simulasi Jarak Dekat (Zoom Macro) & Jarak Jauh (Full Face Portrait)
                T.RandomResizedCrop(224, scale=(0.45, 1.15), ratio=(0.85, 1.15)),
                
                # 2. Simulasi Wajah Hadap Kanan & Kiri (Horizontal Flip + Rotasi Yaw)
                T.RandomHorizontalFlip(p=0.5),
                T.RandomRotation(degrees=(-25, 25)),
                
                # 3. Simulasi Wajah Hadap Atas & Bawah (Perspektif 3D & Distorsi)
                T.RandomPerspective(distortion_scale=0.35, p=0.7),
                T.RandomAffine(degrees=0, translate=(0.08, 0.12), shear=(-12, 12)),
                
                # 4. Simulasi Kondisi Pencahayaan Ruangan (Terang, Hangat, Dingin, Redup)
                T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.08),
                
                # 5. Simulasi Getaran Tangan / Motion Blur saat Scan
                T.RandomApply([T.GaussianBlur(kernel_size=(5, 5), sigma=(0.1, 1.8))], p=0.3),
                
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = T.Compose([
                T.Resize((240, 240)),
                T.CenterCrop(224),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            
    def __call__(self, img):
        return self.transform(img)

# Dataset PyTorch dengan Multi-Task Targets (Skor 0-100 per kondisi)
class MultiTaskSkinDataset(Dataset):
    def __init__(self, master_dir: Path, transform=None):
        self.samples = [] # (filepath, targets_dict)
        self.transform = transform
        
        for cat in CATEGORIES:
            cat_dir = master_dir / cat
            if not cat_dir.exists():
                continue
            for img_path in cat_dir.glob("*.jpg"):
                # Buat target skor kontinu (0.0 - 1.0) untuk 4 dimensi
                # 1.0 = Indikasi sangat tinggi, 0.0 = Bersih / tidak ada indikasi
                targets = {
                    "acne": 0.85 if cat == "acne" else 0.08,
                    "redness": 0.85 if cat == "redness" else (0.35 if cat == "acne" else 0.08),
                    "hyperpigmentation": 0.85 if cat == "hyperpigmentation" else 0.08,
                    "texture": 0.85 if cat == "texture" else (0.50 if cat == "acne" else 0.10)
                }
                if cat == "normal_skin":
                    targets = {"acne": 0.05, "redness": 0.05, "hyperpigmentation": 0.05, "texture": 0.05}
                    
                self.samples.append((img_path, targets))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, targets = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
            
        target_tensor = torch.tensor([
            targets["acne"],
            targets["redness"],
            targets["hyperpigmentation"],
            targets["texture"]
        ], dtype=torch.float32)
        
        return image, target_tensor

full_dataset = MultiTaskSkinDataset(MASTER_DIR, transform=SkinCheckMultiAngleTransforms(is_train=True))
print(f"📊 Total sampel siap latih di Master Dataset: {len(full_dataset)}")

# Visualisasikan Contoh Simulasi Multi-Angle
if len(full_dataset) > 0:
    sample_img_path, _ = full_dataset.samples[0]
    raw_img = Image.open(sample_img_path).convert("RGB")
    aug_pipeline = SkinCheckMultiAngleTransforms(is_train=True)
    
    fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
    axes[0].imshow(raw_img.resize((224, 224)))
    axes[0].set_title("1. Pose Asli (Front)")
    axes[0].axis("off")
    
    titles = ["2. Hadap Kanan/Kiri", "3. Hadap Atas (Tilt)", "4. Hadap Bawah", "5. Jarak Dekat (Zoom)", "6. Jarak Jauh + Cahaya"]
    inv_normalize = T.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    
    for i in range(1, 6):
        t_img = aug_pipeline(raw_img)
        disp_img = inv_normalize(t_img).permute(1, 2, 0).clamp(0, 1).numpy()
        axes[i].imshow(disp_img)
        axes[i].set_title(titles[i-1])
        axes[i].axis("off")
        
    plt.suptitle("🔄 Simulasi Multi-Angle Scan (Pose Invariant Augmentation)", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


## 🧠 Tahap 5: Arsitektur Model AI Multi-Task (MobileNetV3 Backbone)
Model ini memiliki **1 Feature Extractor bersama** dan **4 Task Heads spesifik**:
1. `acne_head` -> Output estimasi skor jerawat (0–100)
2. `redness_head` -> Output estimasi skor kemerahan kulit (0–100)
3. `hyperpigmentation_head` -> Output estimasi flek/pigmentasi (0–100)
4. `texture_head` -> Output estimasi ketidakrataan pori/tekstur (0–100)

Arsitektur ini sangat efisien, cepat diproses di server/browser, dan ideal untuk ekspor ONNX.


In [ ]:
# 5. Definisi Arsitektur Multi-Task Neural Network
import torch.nn as nn
import torchvision.models as models

class SkinCheckMultiTaskNet(nn.Module):
    def __init__(self, pretrained=True):
        super(SkinCheckMultiTaskNet, self).__init__()
        
        # Backbone MobileNetV3-Large (Ringan & Sangat Cepat untuk Web)
        weights = models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        base_model = models.mobilenet_v3_large(weights=weights)
        self.features = base_model.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        feature_dim = 960 # Dimensi output MobileNetV3-Large
        
        # 4 Dedicated Prediction Heads dengan Dropout Regularization
        self.acne_head = self._make_head(feature_dim)
        self.redness_head = self._make_head(feature_dim)
        self.hyperpigmentation_head = self._make_head(feature_dim)
        self.texture_head = self._make_head(feature_dim)
        
    def _make_head(self, in_features):
        return nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Dropout(p=0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid() # Normalisasi ke rentang 0.0 - 1.0
        )
        
    def forward(self, x):
        feat = self.features(x)
        feat = self.avgpool(feat)
        feat = torch.flatten(feat, 1)
        
        # Estimasi 4 kondisi secara independen namun terkoordinasi
        acne_score = self.acne_head(feat) * 100.0
        redness_score = self.redness_head(feat) * 100.0
        pigment_score = self.hyperpigmentation_head(feat) * 100.0
        texture_score = self.texture_head(feat) * 100.0
        
        # Gabungkan dalam tensor (Batch_size, 4)
        output = torch.cat([acne_score, redness_score, pigment_score, texture_score], dim=1)
        return output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SkinCheckMultiTaskNet(pretrained=True).to(device)
print(f"🚀 Model SkinCheckMultiTaskNet berhasil diinisialisasi pada perangkat: {device}")

# Uji forward pass dengan dummy input
dummy_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    test_out = model(dummy_input)
print("✅ Forward test OK! Output shape:", test_out.shape, "(Batch_size, 4 Indikator)")


## 🏋️ Tahap 6: Training Loop dengan Fitur Pause & Resume
Fitur penting:
- 🔁 **Resume Training**: Jika Anda ingin menambah dataset lagi di masa depan, jalankan sel ini kembali. Model akan melanjutkan pembelajaran dari checkpoint sebelumnya (`checkpoint_latest.pth`) tanpa perlu mengulang dari awal!
- 💾 **Best Model Auto-Save**: Otomatis menyimpan bobot terbaik (`best_model.pth`) berdasarkan validation loss terendah.


In [ ]:
# 6. Training Loop dengan Checkpointing & Support Resume
from torch.utils.data import random_split

# Hyperparameter Pelatihan
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
EPOCHS = 10              # Sesuaikan jumlah epoch sesuai kebutuhan
RESUME_TRAINING = True   # True jika ingin melanjutkan dari checkpoint sebelumnya

# Pembagian Train (80%) dan Validation (20%)
val_size = max(1, int(0.2 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

criterion = nn.SmoothL1Loss() # Robust terhadap outlier noise
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

checkpoint_path = CHECKPOINT_DIR / "checkpoint_latest.pth"
best_model_path = CHECKPOINT_DIR / "best_model.pth"

start_epoch = 1
best_val_loss = float("inf")

# Cek apakah ada checkpoint sebelumnya untuk dilanjutkan
if RESUME_TRAINING and checkpoint_path.exists():
    print(f"📂 Ditemukan checkpoint sebelumnya! Memuat dari {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint.get("best_val_loss", float("inf"))
    print(f"⏩ Melanjutkan pelatihan dari Epoch {start_epoch}...")

print(f"🚀 Memulai Training: {len(train_set)} data latih, {len(val_set)} data validasi.")
history = {"train_loss": [], "val_loss": []}

for epoch in range(start_epoch, start_epoch + EPOCHS):
    model.train()
    running_loss = 0.0
    
    for images, targets in train_loader:
        images = images.to(device)
        targets = (targets * 100.0).to(device) # Skala 0-100
        
        optimizer.zero_grad()
        predictions = model(images)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    scheduler.step()
    epoch_train_loss = running_loss / len(train_set)
    
    # Validasi
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            targets = (targets * 100.0).to(device)
            predictions = model(images)
            loss = criterion(predictions, targets)
            val_loss += loss.item() * images.size(0)
            
    epoch_val_loss = val_loss / len(val_set)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    
    print(f"Epoch [{epoch}/{start_epoch + EPOCHS - 1}] -> Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    # Simpan Checkpoint Terkini (Dapat di-resume kapan saja)
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss
    }, checkpoint_path)
    
    # Simpan Bobot Terbaik
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"  ⭐ Model terbaik baru tersimpan dengan Val Loss: {best_val_loss:.4f}")

print("\n🎉 Pelatihan Selesai!")


## 🧪 Tahap 7: Uji Coba Interaktif di Colab (Single Image & Multi-Angle Burst)
Anda dapat mencoba model langsung dengan mengunggah foto wajah Anda sendiri untuk melihat prediksi 4 indikator kondisi kulit serta simulasi multi-sudut.


In [ ]:
# 7. Pengujian Interaktif di Colab dengan Radar Plot & Multi-Angle Simulation
from google.colab import files

def predict_single_image(image_input, model_instance):
    """Menjalankan inferensi dan mengembalikan dictionary skor 0-100."""
    if isinstance(image_input, (str, Path)):
        pil_img = Image.open(image_input).convert("RGB")
    else:
        pil_img = image_input.convert("RGB")
        
    eval_transform = SkinCheckMultiAngleTransforms(is_train=False)
    tensor_img = eval_transform(pil_img).unsqueeze(0).to(device)
    
    model_instance.eval()
    with torch.no_grad():
        preds = model_instance(tensor_img).cpu().numpy()[0]
        
    scores = {
        "acne_score": int(np.clip(preds[0], 0, 100)),
        "redness_score": int(np.clip(preds[1], 0, 100)),
        "hyperpigmentation_score": int(np.clip(preds[2], 0, 100)),
        "texture_score": int(np.clip(preds[3], 0, 100))
    }
    
    # Overall score (skor kesehatan kulit secara keseluruhan, 100 = sempurna bersih)
    penalty = (scores["acne_score"]*0.35 + scores["redness_score"]*0.25 + 
               scores["hyperpigmentation_score"]*0.25 + scores["texture_score"]*0.15)
    scores["overall_score"] = int(np.clip(100 - penalty, 0, 100))
    return scores, pil_img

def plot_skin_report(scores, pil_img):
    """Visualisasikan hasil dalam bentuk visual dashboard yang menawan."""
    fig = plt.figure(figsize=(12, 5))
    
    # Foto Wajah
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.imshow(pil_img)
    ax1.set_title("📸 Foto Pemindaian Wajah", fontsize=12, fontweight='bold')
    ax1.axis("off")
    
    # Grafik Batang Indikator Visual
    ax2 = fig.add_subplot(1, 2, 2)
    categories = ['Jerawat (Acne)', 'Kemerahan (Redness)', 'Hyperpigmentasi', 'Tekstur Kulit']
    vals = [scores['acne_score'], scores['redness_score'], scores['hyperpigmentation_score'], scores['texture_score']]
    colors = ['#e74c3c', '#e67e22', '#9b59b6', '#3498db']
    
    bars = ax2.barh(categories, vals, color=colors, height=0.55)
    ax2.set_xlim(0, 100)
    ax2.set_xlabel("Tingkat Indikasi Visual (0 - 100)")
    ax2.set_title(f"📊 Skor Kondisi Kulit (Overall: {scores['overall_score']}/100)", fontsize=12, fontweight='bold')
    
    for bar in bars:
        w = bar.get_width()
        ax2.text(w + 2, bar.get_y() + bar.get_height()/2, f"{int(w)}%", va='center', fontweight='bold')
        
    plt.tight_layout()
    plt.show()
    
    print("📋 RINGKASAN INDIKATOR VISUAL (NON-MEDIS):")
    print(f" • Jerawat (Acne): {scores['acne_score']}% indication")
    print(f" • Kemerahan (Redness): {scores['redness_score']}% indication")
    print(f" • Hyperpigmentasi / Flek: {scores['hyperpigmentation_score']}% indication")
    print(f" • Tekstur Kulit: {scores['texture_score']}% irregularity indication")
    print(f" • Overall Health Score: {scores['overall_score']}/100")

# Contoh Uji Coba: Ambil 1 sampel dari dataset atau upload foto sendiri
print("💡 Pilih cara pengujian:")
print("1. Gunakan foto dari dataset yang sudah ditimbun")
print("2. Atau upload foto wajah baru")

# Menguji dengan sampel pertama di dataset
if len(full_dataset) > 0:
    test_sample_path, _ = full_dataset.samples[0]
    print(f"\n🧪 Menjalankan inferensi uji coba pada: {test_sample_path.name}")
    results, test_img = predict_single_image(test_sample_path, model)
    plot_skin_report(results, test_img)


## 📦 Tahap 8: One-Click Export ke Antigravity
Sel ini akan mengemas model yang sudah dilatih ke dalam format standar:
1. 🌐 **`skincheck_model.onnx`** (Format portable, sangat cepat dan ringan untuk web).
2. ⚡ **`skincheck_model.pt`** (TorchScript model).
3. 📄 **`model_metadata.json`** (Konfigurasi input size 224x224, nilai normalisasi, dan nama indikator).
4. 🎁 **`skincheck_ai_exported.zip`** (Paket kompresi yang langsung terdownload ke komputer Anda).


In [ ]:
# 8. Ekspor Model ke ONNX, TorchScript & Metadata untuk Antigravity
import os
import json
import shutil
from pathlib import Path
import torch
from google.colab import files

print("📦 Menyiapkan berkas ekspor untuk Antigravity...")

# Pastikan path direktori terdefinisi secara independen
BASE_DIR = Path("/content/skincheck_data")
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
EXPORT_DIR = BASE_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = CHECKPOINT_DIR / "best_model.pth"
checkpoint_latest = CHECKPOINT_DIR / "checkpoint_latest.pth"

# Pastikan instance model tersedia
if 'model' not in globals() or model is None:
    print("ℹ️ Menginisialisasi arsitektur model...")
    model = SkinCheckMultiTaskNet(pretrained=False)

# Muat bobot model terbaik / checkpoint jika ada
if best_model_path.exists():
    print(f"📂 Memuat bobot terbaik dari {best_model_path}...")
    model.load_state_dict(torch.load(best_model_path, map_location="cpu"))
elif checkpoint_latest.exists():
    print(f"📂 Memuat bobot dari checkpoint terbaru: {checkpoint_latest}...")
    checkpoint = torch.load(checkpoint_latest, map_location="cpu")
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict)
else:
    print("ℹ️ Menggunakan bobot model aktif dari memori pelatihan saat ini.")

model.to("cpu")
model.eval()

# 1. Export ke format ONNX
onnx_export_path = EXPORT_DIR / "skincheck_model.onnx"
dummy_cpu_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_cpu_input,
    str(onnx_export_path),
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=["input_image"],
    output_names=["skin_indicators"],
    dynamic_axes={
        "input_image": {0: "batch_size"},
        "skin_indicators": {0: "batch_size"}
    }
)
print(f"✅ Model ONNX berhasil diekspor: {onnx_export_path.name}")

# 2. Export ke TorchScript .pt
pt_export_path = EXPORT_DIR / "skincheck_model.pt"
traced_model = torch.jit.trace(model, dummy_cpu_input)
traced_model.save(str(pt_export_path))
print(f"✅ Model TorchScript berhasil diekspor: {pt_export_path.name}")

# 3. Buat model_metadata.json
metadata = {
    "model_name": "SkinCheck-MultiTask-Net",
    "version": "1.0.0",
    "architecture": "MobileNetV3-Large MultiTask",
    "input_size": [224, 224, 3],
    "input_mean": [0.485, 0.456, 0.406],
    "input_std": [0.229, 0.224, 0.225],
    "indicators": [
        {"name": "acne", "label": "Jerawat", "scale": [0, 100]},
        {"name": "redness", "label": "Kemerahan", "scale": [0, 100]},
        {"name": "hyperpigmentation", "label": "Hyperpigmentasi", "scale": [0, 100]},
        {"name": "texture", "label": "Tekstur Kulit", "scale": [0, 100]}
    ],
    "non_medical_notice": "Estimated cosmetic visual indicators only, not for medical diagnosis."
}

metadata_path = EXPORT_DIR / "model_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print("✅ model_metadata.json berhasil dibuat!")

# 4. Kompres ke ZIP untuk 1-Click Download
zip_output_base = "/content/skincheck_ai_exported"
shutil.make_archive(zip_output_base, 'zip', EXPORT_DIR)
zip_file_path = f"{zip_output_base}.zip"

print(f"🎁 File ZIP siap diunduh: {zip_file_path}")
print("⬇️ Memicu unduhan otomatis ke komputer Anda...")
try:
    files.download(zip_file_path)
    print("✅ Download popup aktif di browser Anda!")
except Exception as e:
    print("ℹ️ Silakan download manual dari tab Files Colab di sebelah kiri: skincheck_ai_exported.zip")
